# GDPR Pipeline Orchestrator

This notebook orchestrates:
1. Policy chunker
2. Chunked text overlap builder
3. REA requirements extractor
4. Vector search
5. Reranker
6. Main clause extractor
7. Graph traversal
8. Prompt sender (generate prompts)
9. Prompt sender (send prompts)


In [1]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path('/Users/my/Documents/projects/detectionDeviation')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Exists:', PROJECT_ROOT.exists())


Project root: /Users/my/Documents/projects/detectionDeviation
Exists: True


## Configuration

In [7]:
# -------------------------------
# Core input/output paths
# -------------------------------
INPUT_REA_WITH_INJECTIONS = PROJECT_ROOT / 'input' / 'rea_case3_injections'
POLICY_FULL_TEXT_FILE = INPUT_REA_WITH_INJECTIONS / 'rea_full_text.txt'
CHUNKED_TEXT_DIR = INPUT_REA_WITH_INJECTIONS / 'chunked_texts'
CHUNKED_POLICY_JSON = CHUNKED_TEXT_DIR / 'chunked_policy.json'

CHUNKED_TEXT_OVERLAP_DIR = INPUT_REA_WITH_INJECTIONS / 'chunked_texts_overlapping_window'

# REA requirements extracted from chunk*.txt files
REA_REQUIREMENTS_ROOT = PROJECT_ROOT / 'intermediate_results' / 'rea_case3_injections'

# REG requirements (for vector DB + graph)
REG_REQUIREMENTS_JSON = (PROJECT_ROOT / 'intermediate_results' / 'reg_eu_ai_act'
                         / 'eu_ai_requirements_with_additional_references.json')

# Vector DB + retrieval outputs
CHROMA_PERSIST_DIR = PROJECT_ROOT / 'pipeline' / 'retrieval' / 'chromadb_store'
ARTIFACT_01_DIR = PROJECT_ROOT / 'intermediate_outputs' / 'artifact_01_rea_case3_injections'
ARTIFACT_01_RERANKED_DIR = PROJECT_ROOT / 'intermediate_outputs' / 'artifact_01_rea_case3_injections_reranked'
ARTIFACT_02_DIR = PROJECT_ROOT / 'intermediate_outputs' / 'artifact_02_rea_case3_injections'
ARTIFACT_03_DIR = PROJECT_ROOT / 'intermediate_outputs' / 'artifact_03_rea_case3_injections'

# Prompt sender / OpenAI config
REASONING_ENV_PATH = PROJECT_ROOT / 'pipeline' / 'reasoning' / '.env'
OPENAI_MODEL = 'gpt-5'

# -------------------------------
# Run flags
# -------------------------------
RUN_POLICY_CHUNKER = True
RUN_OVERLAP_BUILDER = True
RUN_REA_REQUIREMENTS_EXTRACTOR = True
RUN_VECTOR_SEARCH = True
RUN_RERANKER = True
RUN_MAIN_CLAUSE_EXTRACTOR = True
RUN_GRAPH_TRAVERSAL = True
RUN_PROMPT_GENERATION = True
RUN_PROMPT_SENDING = True  # keep False by default to avoid accidental API spend

# -------------------------------
# Pipeline behavior
# -------------------------------
USE_OVERLAP_FOR_REA_REQUIREMENTS = False  # recommended: False for retrieval query quality
BUILD_VECTOR_INDEX = True  # True only if you need to rebuild embeddings

VECTOR_MODEL = 'BAAI/bge-large-en-v1.5'
VECTOR_COLLECTION_NAME = 'gdpr_requirements'
VECTOR_TOP_K = 40

RERANK_MODEL = 'kanon-2-reranker'
RERANK_TOP_N = VECTOR_TOP_K
RERANK_LLM_MAX_ITEMS = None
MAIN_CLAUSE_TOP_K = 5
GRAPH_MAX_HOP = 1

WINDOWED_CHUNK_FOR_PROMPT_SENDER = False  # uses PromptSender's neighbor-window target chunk mode

print('Config loaded.')


Config loaded.


In [8]:
# Evaluation folders
GOLDSTANDARD_ROOT = PROJECT_ROOT / 'goldstandard'

EVAL_EXPERIMENT_NAME = 'rea_case3_injections'
EVAL_EXPERIMENT_DIR = PROJECT_ROOT / 'evaluation' / 'artifact3' / EVAL_EXPERIMENT_NAME
EVAL_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

# Comparison visuals inputs/outputs
EVAL_COMPARE_INPUT_ROOT = PROJECT_ROOT / 'evaluation' / 'artifact3'
EVAL_COMPARE_OUTPUT_DIR = PROJECT_ROOT / 'evaluation' / 'artifact3' / 'compare_reranker_visuals'

RUN_LLM_OUTPUT_EXTRACTION = True
RUN_EVAL_RETRIEVAL = True
RUN_EVAL_ANALYSIS = True
RUN_EVAL_VISUALIZATION = True

print('Eval experiment dir:', EVAL_EXPERIMENT_DIR)


Eval experiment dir: /Users/my/Documents/projects/detectionDeviation/evaluation/artifact3/rea_case3_injections


## Step 1: Policy Chunker (Llama 3)

In [3]:
if RUN_POLICY_CHUNKER:
    from pipeline.retrieval.Llama3PolicyChunker import Llama3PolicyChunker

    CHUNKED_TEXT_DIR.mkdir(parents=True, exist_ok=True)
    chunker = Llama3PolicyChunker(
        endpoint_url='http://localhost:11434/api/chat',
        model_name='llama3',
        timeout_seconds=300,
        max_retries=3,
    )
    saved = chunker.chunk_policy(
        input_file=POLICY_FULL_TEXT_FILE,
        output_file=CHUNKED_POLICY_JSON,
    )
    print('Saved:', saved)
else:
    print('Skipped Step 1')


Sending text to local Llama 3 for intelligent chunking... (attempt 1/3)


JSONDecodeError: Invalid control character at: line 6 column 48 (char 892)

## Step 2: Overlap Builder

In [8]:
if RUN_OVERLAP_BUILDER:
    from pipeline.retrieval.ChunkedTextOverlapBuilder import ChunkedTextOverlapBuilder

    builder = ChunkedTextOverlapBuilder(overlap_hops=1)
    result = builder.build_overlapping_chunks(
        input_dir=CHUNKED_TEXT_DIR,
        output_dir=CHUNKED_TEXT_OVERLAP_DIR,
    )
    print(json.dumps(result, indent=2, ensure_ascii=False))
else:
    print('Skipped Step 2')


{
  "input_dir": "/Users/my/Documents/projects/detectionDeviation/input/rea_case3_injections/chunked_texts",
  "output_dir": "/Users/my/Documents/projects/detectionDeviation/input/rea_case3_injections/chunked_texts_overlapping_window",
  "overlap_hops": 1,
  "files_written": 9,
  "manifest_file": "/Users/my/Documents/projects/detectionDeviation/input/rea_case3_injections/chunked_texts_overlapping_window/overlap_manifest.json"
}


## Step 3: REA Requirements Extractor

In [9]:
if RUN_REA_REQUIREMENTS_EXTRACTOR:
    from pipeline.extractor.ReaRequirementsExtractor import ReaRequirementsExtractor

    source_txt_dir = CHUNKED_TEXT_OVERLAP_DIR if USE_OVERLAP_FOR_REA_REQUIREMENTS else CHUNKED_TEXT_DIR
    saved_paths = ReaRequirementsExtractor.process_folder(
        input_folder=source_txt_dir,
        output_root_folder=REA_REQUIREMENTS_ROOT,
    )
    print(f'Processed {len(saved_paths)} files from {source_txt_dir}')
    for path in saved_paths[:5]:
        print(' -', path)
    if len(saved_paths) > 5:
        print(' ...')
else:
    print('Skipped Step 3')


Processed 9 files from /Users/my/Documents/projects/detectionDeviation/input/rea_case3_injections/chunked_texts
 - /Users/my/Documents/projects/detectionDeviation/intermediate_results/rea_case3_injections/chunk1/chunk1_requirements.json
 - /Users/my/Documents/projects/detectionDeviation/intermediate_results/rea_case3_injections/chunk2/chunk2_requirements.json
 - /Users/my/Documents/projects/detectionDeviation/intermediate_results/rea_case3_injections/chunk3/chunk3_requirements.json
 - /Users/my/Documents/projects/detectionDeviation/intermediate_results/rea_case3_injections/chunk4/chunk4_requirements.json
 - /Users/my/Documents/projects/detectionDeviation/intermediate_results/rea_case3_injections/chunk5/chunk5_requirements.json
 ...


## Step 4: Vector Search

In [13]:
if RUN_VECTOR_SEARCH:
    from pipeline.retrieval.VectorEmbedding import VectorSearch

    search = VectorSearch(
        model_name=VECTOR_MODEL,
        collection_name=VECTOR_COLLECTION_NAME,
    )

    if BUILD_VECTOR_INDEX:
        embed_result = search.embed_and_store(
            input_json_path=REG_REQUIREMENTS_JSON,
            chroma_persist_dir=CHROMA_PERSIST_DIR,
            batch_size=32,
        )
        print('Index build:', json.dumps(embed_result, indent=2, ensure_ascii=False))
    else:
        print('Index build skipped (reusing existing Chroma collection).')

    search_result = search.vector_search_for_rea_folder(
        rea_json_root_path=REA_REQUIREMENTS_ROOT,
        chroma_persist_dir=CHROMA_PERSIST_DIR,
        artifact_root_dir=ARTIFACT_01_DIR,
        top_k=VECTOR_TOP_K,
    )
    print('Vector search:', json.dumps(search_result, indent=2, ensure_ascii=False)[:2000], '...')
else:
    print('Skipped Step 4')


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 9356.94it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index build: {
  "input_count": 79,
  "collection_name": "gdpr_requirements",
  "persist_dir": "/Users/my/Documents/projects/detectionDeviation/pipeline/retrieval/chromadb_store",
  "model_name": "BAAI/bge-large-en-v1.5"
}
Vector search: {
  "chunk_count": 9,
  "top_k": 40,
  "artifact_root_dir": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_case3_injections",
  "chunks": [
    {
      "input_file": "/Users/my/Documents/projects/detectionDeviation/intermediate_results/rea_case3_injections/chunk1/chunk1_requirements.json",
      "chunk_name": "chunk1",
      "rea_count": 3,
      "output_dir": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_case3_injections/chunk1"
    },
    {
      "input_file": "/Users/my/Documents/projects/detectionDeviation/intermediate_results/rea_case3_injections/chunk2/chunk2_requirements.json",
      "chunk_name": "chunk2",
      "rea_count": 4,
      "output_dir": "/Users/my/Document

## Step 5: Reranker

In [9]:
from dotenv import load_dotenv
load_dotenv("/Users/my/Documents/projects/detectionDeviation/pipeline/reasoning/.env")

if RUN_RERANKER:
    from pipeline.retrieval.Reranker import Reranker

    reranker = Reranker(
        model_name=RERANK_MODEL,
        api_key_env='ISAACUS_API_KEY',
    )
    rerank_result = reranker.rerank_artifact_root(
        input_artifact_01_dir=ARTIFACT_01_DIR,
        output_artifact_dir=ARTIFACT_01_RERANKED_DIR,
        top_n=RERANK_TOP_N,
        llm_max_items=RERANK_LLM_MAX_ITEMS,
    )
    print(json.dumps(rerank_result, indent=2, ensure_ascii=False)[:2000], '...')
else:
    print('Skipped Step 5')


{
  "input_root": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_case3_injections",
  "output_root": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_case3_injections_reranked",
  "chunk_count": 9,
  "top_n": 40,
  "llm_max_items": null,
  "chunks": [
    {
      "input_chunk_dir": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_case3_injections/chunk1",
      "output_chunk_dir": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_case3_injections_reranked/chunk1",
      "file_count": 3,
      "files_written": [
        "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_case3_injections_reranked/chunk1/rea-01.json",
        "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_case3_injections_reranked/chunk1/rea-02.json",
        "/Users/my/Documents/projects/detectionDevia

## Step 6: Main Clause Extractor

In [10]:
if RUN_MAIN_CLAUSE_EXTRACTOR:
    from pipeline.retrieval.MainClauseExtractor import MainClauseExtractor

    extractor = MainClauseExtractor()
    result = extractor.extract_main_clauses_for_artifact(
        reranked_artifact_root_dir=ARTIFACT_01_RERANKED_DIR,
        k=MAIN_CLAUSE_TOP_K,
        output_csv_name='reg_main_clauses.csv',
    )
    print(json.dumps(result, indent=2, ensure_ascii=False)[:2000], '...')
else:
    print('Skipped Step 6')


{
  "reranked_artifact_root": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_case3_injections_reranked",
  "chunk_count": 9,
  "k": 5,
  "chunks": [
    {
      "chunk_dir": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_case3_injections_reranked/chunk1",
      "k": 5,
      "source_json_files": 3,
      "selected_main_clauses": 10,
      "output_csv": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_case3_injections_reranked/chunk1/reg_main_clauses.csv",
      "chunk": "chunk1"
    },
    {
      "chunk_dir": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_case3_injections_reranked/chunk2",
      "k": 5,
      "source_json_files": 4,
      "selected_main_clauses": 13,
      "output_csv": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_case3_injections_reranked/chunk2/reg_main_clauses.csv",
      "

## Step 7: Graph Traversal

In [13]:
if RUN_GRAPH_TRAVERSAL:
    from pipeline.retrieval.GraphTraversal import GraphTraversal

    traversal = GraphTraversal(REG_REQUIREMENTS_JSON)
    graph_result = traversal.process_artifact_chunks(
        artifact_01_dir=ARTIFACT_01_RERANKED_DIR,
        artifact_02_dir=ARTIFACT_02_DIR,
        max_hop=GRAPH_MAX_HOP,
    )
    print(json.dumps(graph_result, indent=2, ensure_ascii=False)[:2000], '...')
else:
    print('Skipped Step 7')


{
  "artifact_01_dir": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_case3_injections_reranked",
  "artifact_02_dir": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_02_rea_case3_injections",
  "chunk_count": 9,
  "max_hop": 1,
  "outputs": [
    {
      "chunk": "chunk1",
      "entry_source": "reg_main_clauses.csv",
      "relationships_file": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_02_rea_case3_injections/chunk1/subgraph_relationships.json",
      "texts_file": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_02_rea_case3_injections/chunk1/subgraph_texts.json",
      "visited_nodes_count": 21,
      "relationships_count": 15
    },
    {
      "chunk": "chunk2",
      "entry_source": "reg_main_clauses.csv",
      "relationships_file": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_02_rea_case3_injections/chunk2/su

## Step 8: Prompt Sender (Generate Prompts)

In [14]:
if RUN_PROMPT_GENERATION:
    from pipeline.reasoning.PromptSender import PromptSender

    sender = PromptSender(env_path=REASONING_ENV_PATH, model_name=OPENAI_MODEL)
    generation_result = sender.generate_prompts_for_all_chunks(
        artifact_01_dir=ARTIFACT_01_RERANKED_DIR,
        artifact_02_dir=ARTIFACT_02_DIR,
        artifact_03_dir=ARTIFACT_03_DIR,
        rea_intermediate_root_dir=REA_REQUIREMENTS_ROOT,
        windowed_chunk=WINDOWED_CHUNK_FOR_PROMPT_SENDER,
    )
    print(json.dumps(generation_result, indent=2, ensure_ascii=False)[:2000], '...')
else:
    print('Skipped Step 8')


{
  "artifact_01_dir": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_case3_injections_reranked",
  "artifact_02_dir": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_02_rea_case3_injections",
  "artifact_03_dir": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_03_rea_case3_injections",
  "windowed_chunk": false,
  "max_main_entry_nodes_per_prompt": 3,
  "prompts": [
    {
      "chunk": "chunk1",
      "prompt_index": "1",
      "llm_prompt_preview_file": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_03_rea_case3_injections/chunk1/llm_prompt_preview_01.md"
    },
    {
      "chunk": "chunk1",
      "prompt_index": "2",
      "llm_prompt_preview_file": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_03_rea_case3_injections/chunk1/llm_prompt_preview_02.md"
    },
    {
      "chunk": "chunk1",
      "prompt_index": "3",
   

## Step 9: Prompt Sender (Send Prompts)

In [6]:
if RUN_PROMPT_SENDING:
    from pipeline.reasoning.PromptSender import PromptSender

    sender = PromptSender(env_path=REASONING_ENV_PATH, model_name=OPENAI_MODEL)
    sending_result = sender.send_prompts_for_all_chunks(
        artifact_03_dir=ARTIFACT_03_DIR,
        temperature=None,
    )
    print(json.dumps(sending_result, indent=2, ensure_ascii=False)[:2000], '...')
else:
    print('Skipped Step 9 (RUN_PROMPT_SENDING=False)')


{
  "artifact_03_dir": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_03_rea_case3_injections",
  "responses": [
    {
      "chunk": "chunk1",
      "prompt_file": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_03_rea_case3_injections/chunk1/llm_prompt_preview_01.md",
      "response_file": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_03_rea_case3_injections/chunk1/llm_response_01.json"
    },
    {
      "chunk": "chunk1",
      "prompt_file": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_03_rea_case3_injections/chunk1/llm_prompt_preview_02.md",
      "response_file": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_03_rea_case3_injections/chunk1/llm_response_02.json"
    },
    {
      "chunk": "chunk1",
      "prompt_file": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_03_rea_case3_injection

## Quick Outputs Check

In [16]:
for p in [
    CHUNKED_POLICY_JSON,
    CHUNKED_TEXT_OVERLAP_DIR,
    REA_REQUIREMENTS_ROOT,
    ARTIFACT_01_DIR,
    ARTIFACT_01_RERANKED_DIR,
    ARTIFACT_02_DIR,
    ARTIFACT_03_DIR,
]:
    print(p, '->', p.exists())


/Users/my/Documents/projects/detectionDeviation/input/rea_no_injections/chunked_texts/chunked_policy.json -> False
/Users/my/Documents/projects/detectionDeviation/input/rea_no_injections/chunked_texts_overlapping_window -> True
/Users/my/Documents/projects/detectionDeviation/intermediate_results/rea_no_injections -> True
/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_no_injections -> True
/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_no_injections_reranked -> True
/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_02_no_injections -> True
/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_03_no_injections -> True


## Step 10: LLM Output Extraction

In [8]:
if RUN_LLM_OUTPUT_EXTRACTION:
    from pipeline.extractor.ExtractLLMOutput import ExtractLLMOutput

    extraction_result = ExtractLLMOutput.process_artifact_root(
        artifact_03_root=ARTIFACT_03_DIR,
        response_filename='llm_response.json',
        output_json_filename='llm_extracted_normalized.json',
        output_csv_filename='llm_extracted_table.csv',
        output_md_filename='llm_extracted_readable.md',
    )
    print(json.dumps(extraction_result, indent=2, ensure_ascii=False)[:2000], '...')
else:
    print('Skipped Step 10')


{
  "artifact_03_root": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_03_rea_case3_injections",
  "processed_count": 9,
  "skipped_count": 0,
  "processed": [
    {
      "chunk_dir": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_03_rea_case3_injections/chunk1",
      "response_file": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_03_rea_case3_injections/chunk1/llm_response.json",
      "normalized_json_file": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_03_rea_case3_injections/chunk1/llm_extracted_normalized.json",
      "csv_file": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_03_rea_case3_injections/chunk1/llm_extracted_table.csv",
      "readable_file": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_03_rea_case3_injections/chunk1/llm_extracted_readable.md"
    },
    {
      "chunk_dir": 

## Step 11: Retrieval Evaluation

In [11]:
from evaluation.artifact3.RetrievalConfusionMatrix import RetrievalConfusionMatrix

# choose the artifact root you want to evaluate
system_retrieval_root = ARTIFACT_01_RERANKED_DIR if ARTIFACT_01_RERANKED_DIR.exists() else ARTIFACT_01_DIR

analyzer = RetrievalConfusionMatrix(
    gold_root=GOLDSTANDARD_ROOT,   # still needed by class init
    system_root=system_retrieval_root,
)

article_gold_json = PROJECT_ROOT / "goldstandard" / "relevant_non_relevant_requirements_case3.json"
clause_gold_json = PROJECT_ROOT / "goldstandard" / "eu_ai_injections_relevant_clauses.json"
reg_metadata_json = PROJECT_ROOT / "intermediate_results" / "reg_eu_ai_act" / "eu_ai_requirements_with_additional_references.json"

merged_csv = EVAL_EXPERIMENT_DIR / "merged_main_clauses_deduplicated.csv"
article_json = EVAL_EXPERIMENT_DIR / "retrieval_confusion_matrix_articles.json"
article_md = EVAL_EXPERIMENT_DIR / "retrieval_confusion_matrix_articles.md"
clause_json = EVAL_EXPERIMENT_DIR / "retrieval_confusion_matrix_clauses.json"
clause_md = EVAL_EXPERIMENT_DIR / "retrieval_confusion_matrix_clauses.md"

# 1) merge all chunk-level reg_main_clauses.csv into one deduplicated file
merged_saved = analyzer.build_deduplicated_merged_main_clauses_csv(
    output_csv=merged_csv,
    reg_metadata_json=reg_metadata_json,
)

# 2) compare merged retrieval vs relevant/non-relevant articles JSON
saved_article_json = analyzer.save_article_relevance_json(
    article_gold_json=article_gold_json,
    merged_main_clauses_csv=merged_saved,
    output_json=article_json,
)
saved_article_md = analyzer.save_article_relevance_markdown(
    article_gold_json=article_gold_json,
    merged_main_clauses_csv=merged_saved,
    output_md=article_md,
)

# 3) compare merged retrieval clause column vs relevant clauses JSON
saved_clause_json = analyzer.save_clause_relevance_json(
    clause_gold_json=clause_gold_json,
    merged_main_clauses_csv=merged_saved,
    output_json=clause_json,
)
saved_clause_md = analyzer.save_clause_relevance_markdown(
    clause_gold_json=clause_gold_json,
    merged_main_clauses_csv=merged_saved,
    output_md=clause_md,
)

print("Saved merged CSV:", merged_saved)
print("Saved article-level JSON:", saved_article_json)
print("Saved article-level MD:", saved_article_md)
print("Saved clause-level JSON:", saved_clause_json)
print("Saved clause-level MD:", saved_clause_md)



Saved merged CSV: /Users/my/Documents/projects/detectionDeviation/evaluation/artifact3/rea_case3_injections/merged_main_clauses_deduplicated.csv
Saved article-level JSON: /Users/my/Documents/projects/detectionDeviation/evaluation/artifact3/rea_case3_injections/retrieval_confusion_matrix_articles.json
Saved article-level MD: /Users/my/Documents/projects/detectionDeviation/evaluation/artifact3/rea_case3_injections/retrieval_confusion_matrix_articles.md
Saved clause-level JSON: /Users/my/Documents/projects/detectionDeviation/evaluation/artifact3/rea_case3_injections/retrieval_confusion_matrix_clauses.json
Saved clause-level MD: /Users/my/Documents/projects/detectionDeviation/evaluation/artifact3/rea_case3_injections/retrieval_confusion_matrix_clauses.md


## Step 12: Analysis Evaluation

In [4]:
from evaluation.artifact3.AnalysisConfusionMatrix import AnalysisConfusionMatrix

evaluator = AnalysisConfusionMatrix(
    gold_root=GOLDSTANDARD_ROOT,
    system_root=ARTIFACT_03_DIR,
)

gold_inj_json = PROJECT_ROOT / "goldstandard" / "eu_ai_injections.json"
merged_system_json = EVAL_EXPERIMENT_DIR / "merged_llm_extracted_normalized.json"
merged_eval_json = EVAL_EXPERIMENT_DIR / "analysis_confusion_matrix_merged_injections.json"
merged_eval_md = EVAL_EXPERIMENT_DIR / "analysis_confusion_matrix_merged_injections.md"

merged_saved = evaluator.build_merged_system_output(merged_system_json)
saved_json = evaluator.save_merged_injection_json(gold_inj_json, merged_saved, merged_eval_json)
saved_md = evaluator.save_merged_injection_markdown(gold_inj_json, merged_saved, merged_eval_md)

print("Saved merged system:", merged_saved)
print("Saved merged analysis JSON:", saved_json)
print("Saved merged analysis MD:", saved_md)


Saved merged system: /Users/my/Documents/projects/detectionDeviation/evaluation/artifact3/rea_case3_injections/merged_llm_extracted_normalized.json
Saved merged analysis JSON: /Users/my/Documents/projects/detectionDeviation/evaluation/artifact3/rea_case3_injections/analysis_confusion_matrix_merged_injections.json
Saved merged analysis MD: /Users/my/Documents/projects/detectionDeviation/evaluation/artifact3/rea_case3_injections/analysis_confusion_matrix_merged_injections.md


## Step 13: Visualization

In [ ]:
if RUN_EVAL_VISUALIZATION:
    from evaluation.artifact3.compare_retrieval_confusion_matrices import visualize_from_folder

    viz_result = visualize_from_folder(
        comparison_input_folder=EVAL_COMPARE_INPUT_ROOT,
        output_dir=EVAL_COMPARE_OUTPUT_DIR,
    )
    print('Visualization outputs:')
    print(json.dumps(viz_result, indent=2, ensure_ascii=False))
else:
    print('Skipped Step 13')
